# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print a quick summary of the dataset
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets and their fields via their `@id`s.

In [ ]:
# List all available RecordSets and their fields
print("Available RecordSets and their fields (by @id):\n")
record_sets = [rs for rs in metadata.record_sets]
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    if hasattr(rs, 'name'):
        print(f"   Name: {rs.name}")
    if hasattr(rs, 'fields'):
        print("   Fields:")
        for field in rs.fields:
            print(f"      - {field.id} (name: {getattr(field, 'name', '')})")
    print('')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing the record set and field `@id`s above.

In [ ]:
# Example: Load all record sets into DataFrames
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  - {len(df)} records; columns: {list(df.columns)}\n")

# For further analysis, pick the main/protein record set.
# The FAIR^2 for this dataset typically has one main assay data RecordSet with a name/ID like 'cr:ProteinTable' or similar.
# We'll auto-select the first non-empty DataFrame for analysis:
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break
print(f"Using {main_record_set_id} for downstream analysis.")
print("\nColumns:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate filtering and normalizing for the 'cr:MW_kDa' (molecular weight in kDa) field if present (by `@id`). Otherwise, the first numeric field will be used as an example.

In [ ]:
# Attempt to select a numeric field (by @id) from the main record set
df = dataframes[main_record_set_id]
numeric_field_id = None

# Heuristically search for numeric-like fields (prioritize molecular weight fields)
preferred_field_ids = ['cr:MW_kDa', 'cr:MW', 'cr:coverage', 'cr:NumPeptides', 'cr:pI']
for pfid in preferred_field_ids:
    if pfid in df.columns:
        numeric_field_id = pfid
        break

# If none of the above, find first column that is numeric
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    raise RuntimeError("No numeric field found for EDA.")

print(f"Selected numeric field for analysis: {numeric_field_id}")

# Drop NA for selected field, ensure numeric
df = df.dropna(subset=[numeric_field_id]).copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering: Only consider proteins with molecular weight > threshold (example: 10 kDa)
threshold = 10.0
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field e.g. 'cr:Sample' or similar, if present
group_field_id = None
candidate_group_fields = ['cr:Sample', 'cr:Condition', 'cr:Group', 'cr:Type']
for gid in candidate_group_fields:
    if gid in df.columns:
        group_field_id = gid
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable categorical field found to group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic histogram and boxplot of the selected numeric field
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.title(f'Histogram of {numeric_field_id}')

plt.subplot(1,2,2)
sns.boxplot(x=filtered_df[numeric_field_id])
plt.title(f'Boxplot of {numeric_field_id}')
plt.tight_layout()
plt.show()

# If group field exists, visualize group means
if group_field_id:
    plt.figure(figsize=(8,5))
    sns.barplot(x=grouped_df.index, y=grouped_df.values)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset. We demonstrated how to reference record sets and fields by their `@id`s, load data dynamically, filter and process records, and visualize results. Further analyses can leverage these techniques to perform protein biomarker discovery or cross-sample comparisons as required.